# Fake Review Detector Pro - Google Colab Environment
This notebook sets up the environment, fine-tunes/loads DistilBERT, and launches a Streamlit Web App using Localtunnel.

In [ ]:
!pip install -q transformers streamlit datasets scikit-learn
!npm install localtunnel

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import time
import os
import joblib
from transformers import pipeline, DistilBertTokenizer, DistilBertForSequenceClassification
import torch

st.set_page_config(page_title="Fake Review Detector Pro", page_icon="🕵️‍♂️", layout="wide")

st.title("🕵️‍♂️ Fake Review Detector Pro")
st.markdown("Detect fake reviews using Fine-tuned DistilBERT and Classical ML Models (SVM, Decision Tree).")

@st.cache_resource
def load_models():
    models = {}
    
    # Load DistilBERT (assumes model is fine-tuned and saved in ./distilbert_model)
    try:
        tokenizer = DistilBertTokenizer.from_pretrained("./distilbert_model")
        model = DistilBertForSequenceClassification.from_pretrained("./distilbert_model")
        models['DistilBERT'] = pipeline("text-classification", model=model, tokenizer=tokenizer)
    except:
        models['DistilBERT'] = None 
        
    try:
        models['SVM'] = joblib.load("svm_classifier.joblib")
        models['SVM_Word_TFIDF'] = joblib.load("svm_word_tfidf.joblib")
        models['SVM_Char_TFIDF'] = joblib.load("svm_char_tfidf.joblib")
    except:
        models['SVM'] = None
        
    try:
        models['DecisionTree'] = joblib.load("dt_classifier.joblib")
        models['DT_Word_TFIDF'] = joblib.load("dt_word_tfidf.joblib")
    except:
        models['DecisionTree'] = None
        
    return models

models = load_models()

def predict_classical(model_name, text):
    if model_name == "SVM":
        if models['SVM'] and models['SVM_Word_TFIDF'] and models['SVM_Char_TFIDF']:
            word_features = models['SVM_Word_TFIDF'].transform([text])
            char_features = models['SVM_Char_TFIDF'].transform([text])
            from scipy.sparse import hstack
            features = hstack([word_features, char_features])
            pred = models['SVM'].predict(features)[0]
            prob = models['SVM'].predict_proba(features)[0]
            return "Fake" if pred == 1 else "Real", max(prob)
    elif model_name == "DecisionTree":
        if models['DecisionTree'] and models['DT_Word_TFIDF']:
            features = models['DT_Word_TFIDF'].transform([text])
            pred = models['DecisionTree'].predict(features)[0]
            prob = models['DecisionTree'].predict_proba(features)[0]
            return "Fake" if pred == 1 else "Real", max(prob)
    return "N/A", 0.0

def predict_distilbert(text):
    if models['DistilBERT']:
        res = models['DistilBERT'](text)[0]
        # label format depends on fine-tuning, assuming LABEL_1 is fake
        label = "Fake" if res['label'] == 'LABEL_1' else "Real"
        return label, res['score']
    return "N/A (Model not found)", 0.0

tab1, tab2, tab3 = st.tabs(["Text Input", "CSV Upload", "Mock Data Simulation"])

with tab1:
    st.subheader("Analyze Single Review")
    text_input = st.text_area("Enter review text:", height=150)
    if st.button("Analyze Text"):
        if text_input:
            with st.spinner("Analyzing..."):
                time.sleep(1)
                db_lbl, db_score = predict_distilbert(text_input)
                svm_lbl, svm_score = predict_classical("SVM", text_input)
                dt_lbl, dt_score = predict_classical("DecisionTree", text_input)
                
                col1, col2, col3 = st.columns(3)
                
                # Display metrics with colors
                col1.metric("DistilBERT", db_lbl, f"{db_score:.2%} conf", delta_color="inverse" if db_lbl=="Fake" else "normal")
                col2.metric("Linear SVM", svm_lbl, f"{svm_score:.2%} conf", delta_color="inverse" if svm_lbl=="Fake" else "normal")
                col3.metric("Decision Tree", dt_lbl, f"{dt_score:.2%} conf", delta_color="inverse" if dt_lbl=="Fake" else "normal")
        else:
            st.warning("Please enter some text.")

with tab2:
    st.subheader("Analyze Batch (CSV Upload)")
    uploaded_file = st.file_uploader("Upload CSV File", type=['csv'])
    if uploaded_file:
        df = pd.read_csv(uploaded_file)
        st.write("Preview:", df.head(3))
        text_column = st.selectbox("Select the review text column:", df.columns)
        if st.button("Analyze CSV"):
            with st.spinner("Processing reviews..."):
                results = []
                # limit to 50 for performance during demonstration
                for text in df[text_column].fillna("").values[:50]: 
                    db_lbl, _ = predict_distilbert(text)
                    svm_lbl, _ = predict_classical("SVM", text)
                    dt_lbl, _ = predict_classical("DecisionTree", text)
                    results.append({
                        "Review": text[:100] + "...", 
                        "DistilBERT": db_lbl, 
                        "SVM": svm_lbl, 
                        "DecisionTree": dt_lbl
                    })
                
                res_df = pd.DataFrame(results)
                st.dataframe(res_df, use_container_width=True)
                
                # Show summary charts
                st.subheader("Summary")
                st.bar_chart(res_df[['DistilBERT', 'SVM', 'DecisionTree']].apply(pd.Series.value_counts).fillna(0).T)

with tab3:
    st.subheader("Mock Data Simulation")
    st.write("Quickly test the models with pre-defined mock data.")
    if st.button("Generate & Analyze Mock Data"):
        mock_data = [
            "This product is amazing! Five stars, highly recommended to everyone.",
            "Horrible experience, it broke after one use.",
            "I was paid to write this review but honestly it's an okay product.",
            "Fast shipping and works as expected. Good value for money.",
            "DO NOT BUY THIS. IT IS A SCAM AND FAKE."
        ]
        results = []
        for text in mock_data:
            db_lbl, _ = predict_distilbert(text)
            svm_lbl, _ = predict_classical("SVM", text)
            dt_lbl, _ = predict_classical("DecisionTree", text)
            results.append({
                "Review Text": text, 
                "DistilBERT": db_lbl, 
                "SVM": svm_lbl, 
                "Decision Tree": dt_lbl
            })
        st.table(pd.DataFrame(results))


In [ ]:
import urllib
print("Password/Endpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

!streamlit run app.py & npx localtunnel --port 8501
